 # Temat 1, Zadanie 1 – testy funkcji convert_to_polish_name
   Funkcja zamienia liczbę z przedziału <0, 999 999> na jej polską nazwę.

In [1]:
from src.number_converter import convert_to_polish_name

print(convert_to_polish_name(21))
print(convert_to_polish_name(1000))
print(convert_to_polish_name(22000))

dwadzieścia jeden
tysiąc
dwadzieścia dwa tysiące


In [3]:
%%writefile number_converter_tester.py
import pytest
from src.number_converter import convert_to_polish_name


@pytest.mark.parametrize(
    ("number", "expected"),
    [
        #granica dolna i jedności
        (0, "zero"),
        (1, "jeden"),
        (9, "dziewięć"),
        #nastki 10-19
        (10, "dziesięć"),
        (11, "jedenaście"),
        (19, "dziewiętnaście"),
        #dziesiątki 20-99
        (20, "dwadzieścia"),
        (21, "dwadzieścia jeden"),
        (99, "dziewięćdziesiąt dziewięć"),
        #setki 100-999
        (100, "sto"),
        (101, "sto jeden"),
        (200, "dwieście"),
        (500, "pięćset"),
        (999, "dziewięćset dziewięćdziesiąt dziewięć"),
        #tysiące
        (1000, "tysiąc"),
        (2000, "dwa tysiące"),
        (5000, "pięć tysięcy"),
        (12000, "dwanaście tysięcy"),
        (22000, "dwadzieścia dwa tysiące"),
        #granica górna
        (999999, "dziewięćset dziewięćdziesiąt dziewięć tysięcy dziewięćset dziewięćdziesiąt dziewięć"),
    ],
)
def test_convert_valid_number(number, expected):
    assert convert_to_polish_name(number) == expected


@pytest.mark.parametrize("number", [-1, 1_000_000])
def test_convert_number_outside_supported_range(number):
    with pytest.raises(ValueError):
        convert_to_polish_name(number)

Overwriting number_converter_tester.py


In [4]:
!python -m pytest number_converter_tester.py -v

============================= test session starts ==============================
platform darwin -- Python 3.15.2, pytest-9.2.0, pluggy-1.7.0 -- /Users/yahorandreichykau/Desktop/bjsi/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/yahorandreichykau/Desktop/bjsi
configfile: pytest.ini
plugins: platformdirs-4.14.5, cov-7.1.0, anyio-4.16.0
collected 22 items                                                             

number_converter_tester.py::test_convert_valid_number[0-zero] PASSED     [  4%]
number_converter_tester.py::test_convert_valid_number[1-jeden] PASSED    [  9%]
number_converter_tester.py::test_convert_valid_number[9-dziewi\u0119\u0107] PASSED [ 13%]
number_converter_tester.py::test_convert_valid_number[10-dziesi\u0119\u0107] PASSED [ 18%]
number_converter_tester.py::test_convert_valid_number[11-jedena\u015bcie] PASSED [ 22%]
number_converter_tester.py::test_convert_valid_number[19-dziewi\u0119tna\u015bcie] PASSED [ 27%]
number_converter_tester.py::test_convert_vali

# Temat 1, Zadanie 2 – testy jednostkowe funkcji systemu FastAPI

Wybrane funkcje:

| Funkcja | Działanie | Dlaczego ważna |
|---|---|---|
| `get_authorization_scheme_param(header)` | dzieli nagłówek `Authorization` na schemat i parametr, np. `"Bearer abc"` → `("Bearer", "abc")` | korzystają z niej wszystkie mechanizmy bezpieczeństwa FastAPI |
| `jsonable_encoder(obj)` | zamienia obiekty Pythona (daty, Decimal, modele Pydantic) na dane zgodne z JSON | przechodzi przez nią każda odpowiedź API |

In [5]:
from datetime import date
from decimal import Decimal
from fastapi.encoders import jsonable_encoder
from fastapi.security.utils import get_authorization_scheme_param

print(get_authorization_scheme_param("Bearer abc123"))
print(get_authorization_scheme_param(None))
print(jsonable_encoder(date(2026, 11, 10)))
print(jsonable_encoder(Decimal("10.50")))
print(jsonable_encoder({"kiedy": date(2026, 1, 1), "kwota": Decimal("1.5")}))

('Bearer', 'abc123')
('', '')
2026-11-10
10.5
{'kiedy': '2026-01-01', 'kwota': 1.5}


In [6]:
%%writefile tests/test_fastapi_unit.py
import datetime as dt
import enum
import uuid
from decimal import Decimal

import pytest
from fastapi.encoders import jsonable_encoder
from fastapi.security.utils import get_authorization_scheme_param
from pydantic import BaseModel, Field


#dane testowe (fixture)
class Color(enum.Enum):
    RED = "red"


class User(BaseModel):
    name: str
    email: str = Field(alias="emailAddress")
    age: int | None = None


@pytest.fixture
def sample_user():
    return User(name="Jan", emailAddress="jan@example.com", age=30)


#get_authorization_scheme_param
@pytest.mark.parametrize(
    ("header", "expected"),
    [
        ("Bearer abc123", ("Bearer", "abc123")),          #typowy nagłówek
        ("Basic dXNlcjpwYXNz", ("Basic", "dXNlcjpwYXNz")),  #inny schemat
        ("Bearer", ("Bearer", "")),                       #brak parametru
        ("Bearer a b c", ("Bearer", "a b c")),            #dzieli tylko po 1. spacji
        ("", ("", "")),                                   #pusty nagłówek
        (None, ("", "")),                                 #brak nagłówka
    ],
)
def test_get_authorization_scheme_param(header, expected):
    assert get_authorization_scheme_param(header) == expected


#jsonable_encoder
@pytest.mark.parametrize("value", [1, 1.5, "tekst", True, None])
def test_encoder_keeps_simple_values(value):
    assert jsonable_encoder(value) == value


@pytest.mark.parametrize(
    ("value", "expected"),
    [
        (dt.datetime(2026, 11, 10, 12, 30), "2026-11-10T12:30:00"),
        (dt.date(2026, 11, 10), "2026-11-10"),
        (uuid.UUID("12345678-1234-5678-1234-567812345678"), "12345678-1234-5678-1234-567812345678"),
        (Color.RED, "red"),
        (Decimal("10.50"), 10.5),
        (Decimal("10"), 10),
    ],
)
def test_encoder_converts_special_types(value, expected):
    assert jsonable_encoder(value) == expected


def test_encoder_converts_tuple_and_set_to_list():
    assert jsonable_encoder((1, 2, 3)) == [1, 2, 3]
    assert sorted(jsonable_encoder({3, 1, 2})) == [1, 2, 3]


def test_encoder_encodes_nested_structures():
    data = {"when": dt.date(2026, 1, 1), "items": [Decimal("1.5"), Color.RED]}
    assert jsonable_encoder(data) == {"when": "2026-01-01", "items": [1.5, "red"]}


def test_encoder_model_uses_aliases(sample_user):
    assert jsonable_encoder(sample_user) == {"name": "Jan", "emailAddress": "jan@example.com", "age": 30}


def test_encoder_model_without_aliases(sample_user):
    assert jsonable_encoder(sample_user, by_alias=False) == {"name": "Jan", "email": "jan@example.com", "age": 30}


def test_encoder_include_and_exclude(sample_user):
    assert jsonable_encoder(sample_user, include={"name"}) == {"name": "Jan"}
    assert "age" not in jsonable_encoder(sample_user, exclude={"age"})


def test_encoder_exclude_none():
    data = {"a": 1, "b": None}
    assert jsonable_encoder(data) == {"a": 1, "b": None}
    assert jsonable_encoder(data, exclude_none=True) == {"a": 1}

Overwriting tests/test_fastapi_unit.py


In [7]:
!python -m pytest tests/test_fastapi_unit.py -v

============================= test session starts ==============================
platform darwin -- Python 3.15.2, pytest-9.2.0, pluggy-1.7.0 -- /Users/yahorandreichykau/Desktop/bjsi/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/yahorandreichykau/Desktop/bjsi
configfile: pytest.ini
plugins: platformdirs-4.14.5, cov-7.1.0, anyio-4.16.0
collected 23 items                                                             

tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Bearer abc123-expected0] PASSED [  4%]
tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Basic dXNlcjpwYXNz-expected1] PASSED [  8%]
tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Bearer-expected2] PASSED [ 13%]
tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Bearer a b c-expected3] PASSED [ 17%]
tests/test_fastapi_unit.py::test_get_authorization_scheme_param[-expected4] PASSED [ 21%]
tests/test_fastapi_unit.py::test_get_authorization_scheme_param[None-exp

**Opis:** Testy czarnoskrzynkowe – przygotowane na podstawie dokumentacji FastAPI (wejście i oczekiwane wyjście), bez analizy kodu funkcji.

- `get_authorization_scheme_param` – 6 przypadków: typowy nagłówek, inny schemat, brak parametru, kilka spacji, pusty nagłówek, brak nagłówka (`None`).
- `jsonable_encoder` – typy proste, daty, UUID, Enum, Decimal, krotki i zbiory, struktury zagnieżdżone, model Pydantic (aliasy, `include`, `exclude`, `exclude_none`).
- Wykorzystano **fixture** (`sample_user`) oraz **parametryzację** testów.

**Wynik:** wszystkie 23 testy zakończyły się sukcesem (PASSED).

# Temat 2, Zadanie 1 – pokrycie kodu testami funkcji convert_to_polish_name

Testy z Tematu 1, Zadania 1 uruchomiono na implementacji (`number_converter_white_box2.py`).

In [14]:
from pathlib import Path

plik = Path("number_converter_tester.py")
tekst = plik.read_text(encoding="utf-8")
tekst = tekst.replace("from src.number_converter import", "from number_converter_white_box2 import")
plik.write_text(tekst, encoding="utf-8")

print(tekst.splitlines()[1])

from number_converter_white_box2 import convert_to_polish_name


In [15]:
!python -m coverage run -m pytest number_converter_tester.py -q
!python -m coverage report -m

..............F.......                                                   [100%]
=================================== FAILURES ===================================
_________________ test_convert_valid_number[1000-tysi\u0105c] __________________

number = 1000, expected = 'tysiąc'

    @pytest.mark.parametrize(
        ("number", "expected"),
        [
            #granica dolna i jedności
            (0, "zero"),
            (1, "jeden"),
            (9, "dziewięć"),
            #nastki 10-19
            (10, "dziesięć"),
            (11, "jedenaście"),
            (19, "dziewiętnaście"),
            #dziesiątki 20-99
            (20, "dwadzieścia"),
            (21, "dwadzieścia jeden"),
            (99, "dziewięćdziesiąt dziewięć"),
            #setki 100-999
            (100, "sto"),
            (101, "sto jeden"),
            (200, "dwieście"),
            (500, "pięćset"),
            (999, "dziewięćset dziewięćdziesiąt dziewięć"),
            #tysiące
            (1000, "tysiąc"),
 

In [17]:
!COVERAGE_CORE=pytrace python -m pytest number_converter_tester.py --cov=number_converter_white_box2 --cov-branch --cov-context=test -q
!python -m coverage json --show-contexts -o coverage_converter.json
!python tools/function_coverage.py coverage_converter.json number_converter_white_box2.py:convert_to_polish_name

..............F.......                                                   [100%]
=================================== FAILURES ===================================
_________________ test_convert_valid_number[1000-tysi\u0105c] __________________

number = 1000, expected = 'tysiąc'

    @pytest.mark.parametrize(
        ("number", "expected"),
        [
            #granica dolna i jedności
            (0, "zero"),
            (1, "jeden"),
            (9, "dziewięć"),
            #nastki 10-19
            (10, "dziesięć"),
            (11, "jedenaście"),
            (19, "dziewiętnaście"),
            #dziesiątki 20-99
            (20, "dwadzieścia"),
            (21, "dwadzieścia jeden"),
            (99, "dziewięćdziesiąt dziewięć"),
            #setki 100-999
            (100, "sto"),
            (101, "sto jeden"),
            (200, "dwieście"),
            (500, "pięćset"),
            (999, "dziewięćset dziewięćdziesiąt dziewięć"),
            #tysiące
            (1000, "tysiąc"),
 

In [29]:
import coverage, importlib
import number_converter_white_box2 as m

cov = coverage.Coverage(include=["*number_converter_white_box2.py"])
cov.start()
importlib.reload(m)                       #wczytujemy kod pod obserwacją coverage
for liczba in range(0, 1_000_000):        #wszystkie liczby z przedziału
    m.convert_to_polish_name(liczba)
cov.stop()
cov.report(show_missing=True)

Name                             Stmts   Miss  Cover   Missing
--------------------------------------------------------------
number_converter_white_box2.py      37      2    95%   3, 25
--------------------------------------------------------------
TOTAL                               37      2    95%


94.5945945945946

**Weryfikacja:** funkcję uruchomiono dla **wszystkich** liczb z przedziału <0, 999 999>. Linia 25 nadal nie została wykonana – to potwierdza, że jest **kodem nieosiągalnym**. (Linia 3 nie została tu wykonana tylko dlatego, że w tym eksperymencie nie podano liczb spoza zakresu – testy -1 i 1 000 000 ją pokrywają.)

## Analiza wyników

**Wynik testów:** 21 z 22 testów przeszło. Test dla **1000 nie przeszedł** – funkcja zwraca `"jeden tysiąc"` zamiast poprawnego `"tysiąc"` (linia 55). To **błąd implementacji** wykryty przez test (dotyczy też liczb 1001–1999). Linię 55 wykonuje tylko ten jeden test – bez wartości granicznej 1000 błąd nie zostałby wykryty.

**Pokrycie:** **97%** (37 instrukcji, 1 niepokryta). Niepokryta jest **linia 25** (`return ""` dla `n == 0`).

**Test z największym pokryciem:** **999999** (81%, 30 z 37 instrukcji) – przechodzi przez tysiące, setki, dziesiątki i jedności. Najmniej wnoszą -1 i 1 000 000 (kończą się w pierwszych liniach).

**Linia 25 – kod nieosiągalny:** funkcja `three_digits_to_words` jest wywoływana tylko dla `thousands >= 2` i `remainder > 0`, więc nigdy nie dostaje 0. Żaden test zgodny ze specyfikacją jej nie pokryje.

## Wnioski
- Testy oparte na klasach równoważności i wartościach granicznych dały prawie pełne pokrycie i wykryły błąd.
- 100% pokrycia nie zawsze jest możliwe – część kodu bywa nieosiągalna.
- Pokrycie ≠ brak błędów: linia 55 była wykonana, a błąd wykrył dopiero poprawny oczekiwany wynik.

# Temat 2, Zadanie 2 – pokrycie i kontrolowane zmiany w funkcjach FastAPI

Wykorzystano testy z Tematu 1, Zadania 2 (`tests/test_fastapi_unit.py`) dla funkcji:
`get_authorization_scheme_param` oraz `jsonable_encoder` (wraz z pomocniczą `decimal_encoder`, której `jsonable_encoder` używa dla typu `Decimal`).

## Część A – pokrycie instrukcji i gałęzi

In [18]:
!COVERAGE_CORE=pytrace python -m pytest tests/test_fastapi_unit.py --cov=fastapi --cov-branch --cov-context=test --cov-report= -q
!python -m coverage json --show-contexts -o coverage_fastapi.json

.......................                                                  [100%]
23 passed in 0.56s
Wrote JSON report to coverage_fastapi.json


In [19]:
!python tools/function_coverage.py coverage_fastapi.json fastapi/security/utils.py:get_authorization_scheme_param fastapi/encoders.py:decimal_encoder

fastapi/security/utils.py:get_authorization_scheme_param   (linie 1-7 pliku .venv/lib/python3.15/site-packages/fastapi/security/utils.py)
Instrukcje: 5   pokryte: 5   NIEPOKRYTE: 0   pokrycie: 100.0%
Niepokryte linie: -
Niepokryte gałęzie (0):

Pokrycie uzyskiwane przez POJEDYNCZE testy (od największego):
     3/5   ( 60.0%)  tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Basic dXNlcjpwYXNz-expected1]
     3/5   ( 60.0%)  tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Bearer a b c-expected3]
     3/5   ( 60.0%)  tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Bearer abc123-expected0]
     3/5   ( 60.0%)  tests/test_fastapi_unit.py::test_get_authorization_scheme_param[Bearer-expected2]
     2/5   ( 40.0%)  tests/test_fastapi_unit.py::test_get_authorization_scheme_param[-expected4]
     2/5   ( 40.0%)  tests/test_fastapi_unit.py::test_get_authorization_scheme_param[None-expected5]

Które testy wykonują poszczególne linie:
       1 [ 0 tes

In [20]:
!python tools/function_coverage.py coverage_fastapi.json fastapi/encoders.py:jsonable_encoder

fastapi/encoders.py:jsonable_encoder   (linie 129-366 pliku .venv/lib/python3.15/site-packages/fastapi/encoders.py)
Instrukcje: 63   pokryte: 33   NIEPOKRYTE: 30   pokrycie: 52.4%
Niepokryte linie: 233-234, 236-238, 240, 242, 260-262, 276, 280, 285, 287, 338-342, 346-356
     233 |         if type(obj) in custom_encoder:
     234 |             return custom_encoder[type(obj)](obj)
     236 |             for encoder_type, encoder_instance in custom_encoder.items():
     237 |                 if isinstance(obj, encoder_type):
     238 |                     return encoder_instance(obj)
     240 |         include = set(include)  # type: ignore[assignment]  # ty: ignore[invalid-assignment]
     242 |         exclude = set(exclude)  # type: ignore[assignment]  # ty: ignore[invalid-assignment]
     260 |         assert not isinstance(obj, type)
     261 |         obj_dict = dataclasses.asdict(obj)
     262 |         return jsonable_encoder(
     276 |         return str(obj)
     280 |       

### Analiza pokrycia

| Funkcja | Instrukcje | Niepokryte | Pokrycie |
|---|---|---|---|
| `get_authorization_scheme_param` | 5 | 0 | 100% |
| `decimal_encoder` | 5 | 0 | 100% |
| `jsonable_encoder` | 63 | 30 | 52% |
| **Razem** | **73** | **30** | **59%** |

**Łącznie nie pokryto 30 linii** – wszystkie w `jsonable_encoder`: obsługa `custom_encoder` (233–238), zamiana `include`/`exclude` na zbiór (240, 242), dataclass (260–262), `str(obj)` (276), `include`/`exclude` dla słownika (285, 287) oraz końcowe sposoby kodowania i błąd dla nieznanego obiektu (338–356).

**Testy odpowiedzialne za gałęzie:**
- `get_authorization_scheme_param`: pusty nagłówek (linia 5) – testy `""` i `None`; niepusty (linie 6–7) – 4 testy z nagłówkiem.
- `decimal_encoder`: liczba całkowita → `int` (linia 79) – **tylko test `Decimal("10")`**; ułamek → `float` (linia 81) – `Decimal("10.50")` i struktura zagnieżdżona.

**Testy wykonujące te same fragmenty:** 4 testy z nagłówkiem przechodzą identyczną ścieżkę (linie 4, 6, 7), podobnie testy `""` i `None`; testy daty i UUID przechodzą tą samą ścieżką w `jsonable_encoder` (wyszukanie kodera po typie).

**Kod nieosiągalny:** brak – wszystkie niepokryte fragmenty da się wykonać odpowiednimi danymi (np. dataclass, `custom_encoder`). Brak pokrycia to **luka w testach**, a nie kod martwy.

## Część B – kontrolowane zmiany w kodzie

Po każdej zmianie uruchamiany jest cały zestaw testów, a następnie przywracana jest oryginalna implementacja.

| ID | Funkcja | Zmiana | Rodzaj zmiany |
|---|---|---|---|
| M1 | `get_authorization_scheme_param` | `if not header:` → `if header:` | odwrócenie warunku |
| M2 | `get_authorization_scheme_param` | `partition(" ")` → `partition(":")` | zmiana wartości |
| M3 | `get_authorization_scheme_param` | `return scheme, param` → `return param, scheme` | zmiana zwracanej wartości |
| M4 | `get_authorization_scheme_param` | `if not header:` → `if header is None:` | zmiana warunku |
| M5 | `decimal_encoder` | `exponent >= 0` → `exponent > 0` | zmiana wartości granicznej (`>=` na `>`) |
| M6 | `decimal_encoder` | `and exponent >= 0` → `or exponent >= 0` | zamiana `and` na `or` |

In [25]:
import os, subprocess, sys
from pathlib import Path
import fastapi.encoders, fastapi.security.utils

PLIKI = {
    "utils": Path(fastapi.security.utils.__file__),
    "encoders": Path(fastapi.encoders.__file__),
}

def sprawdz_zmiane(nazwa, plik, przed, po, testy=("tests/test_fastapi_unit.py",)):
    sciezka = PLIKI[plik]
    oryginal = sciezka.read_text(encoding="utf-8")
    if przed not in oryginal:
        print(f"{nazwa}: fragmentu nie znaleziono - pominięto\n")
        return
    sciezka.write_text(oryginal.replace(przed, po, 1), encoding="utf-8")   # 1. wprowadzenie zmiany
    try:
        env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1")
        wynik = subprocess.run([sys.executable, "-m", "pytest", *testy, "-q", "-rf", "--color=no", "-p", "no:cacheprovider"],
                               capture_output=True, text=True, env=env)       # 2. cały zestaw testów
    finally:
        sciezka.write_text(oryginal, encoding="utf-8")                        # 3. przywrócenie oryginału
    upadle = [l[len("FAILED "):].split(" - ")[0].split("::")[-1]
              for l in wynik.stdout.splitlines() if l.startswith("FAILED")]
    status = "WYKRYTA" if wynik.returncode != 0 else "NIEWYKRYTA"
    print(f"{nazwa}: {przed!r} -> {po!r}")
    print(f"   {status} | {wynik.stdout.strip().splitlines()[-1]}")
    for t in upadle:
        print(f"     wykrył: {t}")
    print()

In [26]:
sprawdz_zmiane("M1", "utils", "if not authorization_header_value:", "if authorization_header_value:")
sprawdz_zmiane("M2", "utils", 'partition(" ")', 'partition(":")')
sprawdz_zmiane("M3", "utils", "return scheme, param.strip()", "return param.strip(), scheme")
sprawdz_zmiane("M4", "utils", "if not authorization_header_value:", "if authorization_header_value is None:")
sprawdz_zmiane("M5", "encoders", "exponent >= 0", "exponent > 0")
sprawdz_zmiane("M6", "encoders", "and exponent >= 0", "or exponent >= 0")

M1: 'if not authorization_header_value:' -> 'if authorization_header_value:'
   WYKRYTA | 5 failed, 18 passed in 0.19s
     wykrył: test_get_authorization_scheme_param[Bearer abc123-expected0]
     wykrył: test_get_authorization_scheme_param[Basic dXNlcjpwYXNz-expected1]
     wykrył: test_get_authorization_scheme_param[Bearer-expected2]
     wykrył: test_get_authorization_scheme_param[Bearer a b c-expected3]
     wykrył: test_get_authorization_scheme_param[None-expected5]

M2: 'partition(" ")' -> 'partition(":")'
   WYKRYTA | 3 failed, 20 passed in 0.15s
     wykrył: test_get_authorization_scheme_param[Bearer abc123-expected0]
     wykrył: test_get_authorization_scheme_param[Basic dXNlcjpwYXNz-expected1]
     wykrył: test_get_authorization_scheme_param[Bearer a b c-expected3]

M3: 'return scheme, param.strip()' -> 'return param.strip(), scheme'
   WYKRYTA | 4 failed, 19 passed in 0.16s
     wykrył: test_get_authorization_scheme_param[Bearer abc123-expected0]
     wykrył: test_get_autho

In [27]:
%%writefile tests/test_fastapi_unit_extra.py
from decimal import Decimal
from fastapi.encoders import jsonable_encoder

def test_decimal_without_fraction_is_int():
    wynik = jsonable_encoder(Decimal("10"))
    assert wynik == 10
    assert type(wynik) is int      #sprawdzamy także TYP wyniku

Overwriting tests/test_fastapi_unit_extra.py


In [28]:
sprawdz_zmiane("M5 (po dodaniu testu)", "encoders", "exponent >= 0", "exponent > 0",
               testy=("tests/test_fastapi_unit.py", "tests/test_fastapi_unit_extra.py"))

M5 (po dodaniu testu): 'exponent >= 0' -> 'exponent > 0'
   WYKRYTA | 1 failed, 23 passed in 0.17s
     wykrył: test_decimal_without_fraction_is_int



# Temat 3, Zadanie 1 – scenariusze testów integracyjnych

**Badana aplikacja:** Booking API (`app_example/main.py`) zbudowana na FastAPI – rezerwacja sal.

**Współpracujące elementy:**
- routing FastAPI (endpointy `/bookings`, `/rooms/{id}/availability`),
- walidacja danych (Pydantic) – błędne dane → odpowiedź 422,
- moduł rezerwacji (`BookingRepository`) – przechowuje rezerwacje,
- moduł dostępności (`AvailabilityService`) – korzysta z tych samych danych co moduł rezerwacji,
- powiadomienia (`Notifier`) – „wysyła e-mail” po rezerwacji.

| ID | Typ | Stan początkowy | Dane wejściowe | Kolejne działania | Oczekiwany wynik | Gdzie obserwować integrację |
|---|---|---|---|---|---|---|
| S1 | happy path | brak rezerwacji | sala 1, Jan Kowalski, 2026-12-24 | sprawdź dostępność → zarezerwuj → sprawdź dostępność → odczytaj → anuluj → sprawdź dostępność | true → 201 → false → 200 → 204 → true | odpowiedź modułu dostępności, komunikat `[EMAIL]` |
| S2 | błąd | sala 2 zarezerwowana na 2026-12-24 | ta sama sala i dzień, inny gość | zarezerwuj ponownie | 409, druga rezerwacja nie zapisana | kod odpowiedzi, brak rezerwacji nr 2 |
| S3 | niepoprawne dane | brak rezerwacji | `room_id = "abc"` | zarezerwuj | 422, dane nie trafiają do modułu rezerwacji | treść błędu, dostępność bez zmian |
| S4 | błąd | brak rezerwacji | sala 99 (nie istnieje) | zarezerwuj | 404 | kod odpowiedzi |

In [31]:
from fastapi.testclient import TestClient
import app_example.main as aplikacja

klient = TestClient(aplikacja.app)

def reset():
    aplikacja._repository = aplikacja.BookingRepository()   #stan początkowy - brak rezerwacji

def pokaz(krok, odp):
    if odp.status_code == 422:
        tresc = [blad["msg"] for blad in odp.json()["detail"]]
    else:
        tresc = odp.json() if odp.content else ""
    print(f"{krok:<28} -> {odp.status_code} {tresc}")

/Users/yahorandreichykau/Desktop/bjsi/.venv/lib/python3.15/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [32]:
reset()
DZIEN = {"day": "2026-12-24"}
pokaz("1. dostępność (start)", klient.get("/rooms/1/availability", params=DZIEN))
pokaz("2. rezerwacja", klient.post("/bookings", json={"room_id": 1, "guest": "Jan Kowalski", "day": "2026-12-24"}))
pokaz("3. dostępność po rezerwacji", klient.get("/rooms/1/availability", params=DZIEN))
pokaz("4. odczyt rezerwacji", klient.get("/bookings/1"))
pokaz("5. anulowanie", klient.delete("/bookings/1"))
pokaz("6. dostępność po anulowaniu", klient.get("/rooms/1/availability", params=DZIEN))

1. dostępność (start)        -> 200 {'room_id': 1, 'day': '2026-12-24', 'available': True}
[EMAIL] Potwierdzenie rezerwacji #1 dla Jan Kowalski: pokój 1, dzień 2026-12-24
2. rezerwacja                -> 201 {'room_id': 1, 'guest': 'Jan Kowalski', 'day': '2026-12-24', 'id': 1}
3. dostępność po rezerwacji  -> 200 {'room_id': 1, 'day': '2026-12-24', 'available': False}
4. odczyt rezerwacji         -> 200 {'room_id': 1, 'guest': 'Jan Kowalski', 'day': '2026-12-24', 'id': 1}
5. anulowanie                -> 204 
6. dostępność po anulowaniu  -> 200 {'room_id': 1, 'day': '2026-12-24', 'available': True}


In [33]:
reset()
print("S2 – podwójna rezerwacja")
pokaz("rezerwacja Anna", klient.post("/bookings", json={"room_id": 2, "guest": "Anna", "day": "2026-12-24"}))
pokaz("rezerwacja Piotr (ta sama)", klient.post("/bookings", json={"room_id": 2, "guest": "Piotr", "day": "2026-12-24"}))
pokaz("odczyt rezerwacji nr 2", klient.get("/bookings/2"))

reset()
print("\nS3 – niepoprawne dane")
pokaz("rezerwacja room_id='abc'", klient.post("/bookings", json={"room_id": "abc", "guest": "Jan", "day": "2026-12-24"}))
pokaz("dostępność sali 1", klient.get("/rooms/1/availability", params={"day": "2026-12-24"}))

reset()
print("\nS4 – sala nie istnieje")
pokaz("rezerwacja sali 99", klient.post("/bookings", json={"room_id": 99, "guest": "Jan", "day": "2026-12-24"}))

S2 – podwójna rezerwacja
[EMAIL] Potwierdzenie rezerwacji #1 dla Anna: pokój 2, dzień 2026-12-24
rezerwacja Anna              -> 201 {'room_id': 2, 'guest': 'Anna', 'day': '2026-12-24', 'id': 1}
rezerwacja Piotr (ta sama)   -> 409 {'detail': 'Pokój jest już zarezerwowany w tym dniu'}
odczyt rezerwacji nr 2       -> 404 {'detail': 'Rezerwacja nie istnieje'}

S3 – niepoprawne dane
rezerwacja room_id='abc'     -> 422 ['Input should be a valid integer, unable to parse string as an integer']
dostępność sali 1            -> 200 {'room_id': 1, 'day': '2026-12-24', 'available': True}

S4 – sala nie istnieje
rezerwacja sali 99           -> 404 {'detail': 'Pokój 99 nie istnieje'}


### Wyniki testów manualnych

Scenariusze wykonano krok po kroku, wysyłając żądania do aplikacji i porównując odpowiedzi z oczekiwanymi.

| ID | Wynik oczekiwany | Wynik rzeczywisty | Zgodność |
|---|---|---|---|
| S1 | true → 201 → false → 200 → 204 → true | true → 201 → false → 200 → 204 → true, komunikat `[EMAIL]` | Tak |
| S2 | 201, potem 409, rezerwacja nr 2 nie istnieje | 201 → 409 → 404 | Tak |
| S3 | 422, dostępność bez zmian | 422 → true | Tak |
| S4 | 404 | 404 | Tak |

### Błędy
W testach nie wystąpiły błędy. Potencjalne błędy integracji:

| Komponent | Komunikacja | Dane / konfiguracja | Potencjalny błąd | Naprawa |
|---|---|---|---|---|
| AvailabilityService + BookingRepository | sprawdzenie i zapis to dwa osobne kroki | dwie równoczesne rezerwacje tej samej sali | obie mogą zostać zapisane (race condition) | blokada lub unikalne ograniczenie w bazie |
| walidacja Pydantic | JSON → model | `"day": "2020-01-01"` | przyjmowane są rezerwacje w przeszłości | walidator daty |
| BookingRepository | dane w pamięci | restart serwera | utrata wszystkich rezerwacji | baza danych |

# Temat 3, Zadanie 2 – Wariant A: automatyzacja testu integracyjnego

**Uzasadnienie:** FastAPI jest frameworkiem backendowym bez interfejsu użytkownika, więc testy E2E w przeglądarce (Wariant B) nie mają sensu. Automatywano testy integracyjne z Zadania 1.

**Narzędzia:** pytest (fixtures, parametryzacja) + `TestClient` z FastAPI + `app.dependency_overrides` (podmiana zależności).

**Uruchomienie:** `pytest tests/test_integration_booking.py -v`

In [34]:
!cat tests/test_integration_booking.py

"""Temat 3, Zadanie 2 (Wariant A) - automatyzacja testów integracyjnych.

Badana integracja: routing FastAPI + walidacja Pydantic + Depends
+ moduł rezerwacji (BookingRepository) + moduł dostępności (AvailabilityService)
+ powiadomienia (Notifier).

Elementy RZECZYWISTE: aplikacja FastAPI, routing, walidacja, mechanizm Depends,
                      BookingRepository, AvailabilityService.
Elementy zastąpione ATRAPĄ: Notifier -> FakeNotifier (nie wysyłamy e-maili,
                      tylko zapisujemy wywołania, żeby móc je sprawdzić).
Izolacja: każdy test dostaje NOWE, puste repozytorium (fixture `repository`),
          a po teście nadpisania zależności są usuwane -> testy można uruchamiać
          wielokrotnie i w dowolnej kolejności.
"""
import pytest
from fastapi.testclient import TestClient

from app_example.main import BookingRepository, app, get_notifier, get_repository

DAY = "2026-12-24"


class FakeNotifier:
    """Atrapa (fake) usługi e-mail - zapamiętuje, o czym miała pow

In [35]:
!python -m pytest tests/test_integration_booking.py -v

============================= test session starts ==============================
platform darwin -- Python 3.15.2, pytest-9.2.0, pluggy-1.7.0 -- /Users/yahorandreichykau/Desktop/bjsi/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/yahorandreichykau/Desktop/bjsi
configfile: pytest.ini
plugins: platformdirs-4.14.5, cov-7.1.0, anyio-4.16.0
collected 11 items                                                             

tests/test_integration_booking.py::test_s1_happy_path_booking_lifecycle PASSED [  9%]
tests/test_integration_booking.py::test_s2_double_booking_is_rejected PASSED [ 18%]
tests/test_integration_booking.py::test_s3_invalid_data_does_not_reach_repository[room-not-int] PASSED [ 27%]
tests/test_integration_booking.py::test_s3_invalid_data_does_not_reach_repository[bad-date] PASSED [ 36%]
tests/test_integration_booking.py::test_s3_invalid_data_does_not_reach_repository[empty-guest] PASSED [ 45%]
tests/test_integration_booking.py::test_s3_invalid_data_does_not_reach_repos

**Wynik:** wszystkie 11 testów zakończyło się sukcesem.

| Wymaganie | Realizacja |
|---|---|
| przygotowanie stanu | fixture `repository` – nowe, puste repozytorium dla każdego testu |
| sprawdzenie integracji | kody HTTP, zawartość repozytorium, odpowiedź modułu dostępności, wywołania powiadomień |
| jednoznaczne asercje | np. `status_code == 409`, `repository.size() == 1` |
| ponowne uruchomienie | każdy test ma własne dane – można uruchamiać wielokrotnie |
| sprzątanie | `app.dependency_overrides.clear()` po każdym teście |

**Elementy rzeczywiste:** FastAPI (routing, walidacja, Depends), `BookingRepository`, `AvailabilityService`.
**Atrapa:** `Notifier` (e-mail) → `FakeNotifier` – zapisuje wywołania zamiast wysyłać wiadomości.

**Ograniczenia:** brak prawdziwego serwera i sieci (TestClient), brak prawdziwej bazy danych, testy nie sprawdzają równoczesnych rezerwacji. (Ostrzeżenie `StarletteDeprecationWarning` dotyczy tylko zalecanej w przyszłości biblioteki `httpx2` i nie wpływa na wynik testów.)